<style>
.jp-CodeCell .jp-InputArea, div.input_area {background:#eef8ee !important; border-left:5px solid #2e7d32 !important;}
.jp-CodeCell pre, div.input_area pre {color:#145a20 !important;}
.note-box {background:#eef5ff; border-left:5px solid #2f5597; padding:12px; margin:10px 0;}
.warning-box {background:#fff4e5; border-left:5px solid #d97706; padding:12px; margin:10px 0;}
.success-box {background:#edf7ed; border-left:5px solid #2e7d32; padding:12px; margin:10px 0;}
.decision-box {background:#f7f4ff; border-left:5px solid #6b46c1; padding:12px; margin:10px 0;}
</style>

# SBI 812: Machine Learning in Bioinformatics
## Lecture 3 — Building a Proper Machine-Learning Workflow

**Duration:** 3 hours  
**Level:** Introductory  
**Prerequisite:** Lecture 2 — exploration and preprocessing  
**Practical outcome:** Build, evaluate and explain a leakage-aware classification workflow

This lecture uses:

```text
lecture03_bacterial_response.csv
```

Place the CSV file in the same directory as this notebook.

## 1. Why this lecture matters

Training a classifier requires more than calling a modelling function. A defensible workflow must answer several questions:

- What is the prediction target?
- Which columns are legitimate features?
- Which rows are used for learning?
- Which rows are kept unseen for evaluation?
- How are missing values and categories handled?
- Is the model better than a simple baseline?
- Does the reported performance describe unseen data or only memorized training data?

The order of operations matters. If information from the test data influences preprocessing or model fitting, evaluation can become overly optimistic.

> **Define → separate → split → preprocess → fit → predict → evaluate → interpret**

<div class="warning-box"><b>A model that accidentally sees the answers can appear excellent while learning nothing useful.</b></div>

## Learning outcomes

By the end of the lecture, you should be able to:

1. distinguish observations, features, targets and identifiers;
2. create `X` and `y` without including forbidden columns;
3. explain data leakage with biological examples;
4. divide data into training and test sets;
5. explain `test_size`, `random_state` and `stratify`;
6. calculate a simple majority-class baseline;
7. identify numerical and categorical feature groups;
8. use imputers, one-hot encoding and scaling inside a pipeline;
9. explain the roles of `ColumnTransformer` and `Pipeline`;
10. fit preprocessing only from training data;
11. generate predictions for unseen test samples;
12. interpret accuracy, confusion matrices, precision and recall;
13. compare training and test performance cautiously;
14. recognize introductory signs of overfitting; and
15. document a reproducible modelling workflow.

# Part 1 — Import the required tools

The workflow uses pandas, Matplotlib, Seaborn and several tools from scikit-learn.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

sns.set_theme(style="whitegrid")
pd.set_option("display.max_columns", None)

### How to read the imports

The form:

```python
from sklearn.model_selection import train_test_split
```

means: open the `model_selection` part of scikit-learn and import the function named `train_test_split`.

The tools have different jobs:

| Tool | Job |
|---|---|
| `train_test_split` | Divide rows into training and test sets |
| `SimpleImputer` | Replace missing numerical values using a learned rule |
| `OneHotEncoder` | Convert category labels into indicator columns |
| `StandardScaler` | Put numerical features on comparable scales |
| `ColumnTransformer` | Apply different transformations to different columns |
| `Pipeline` | Connect preprocessing and a model in one ordered workflow |
| `DummyClassifier` | Provide a deliberately simple baseline |
| `LogisticRegression` | Fit an introductory classification model |
| Metrics functions | Compare predictions with known test labels |

Importing a model does not fit it. Each imported name only makes a tool available.

# Part 2 — Load and inspect the dataset

In [ ]:
filename = "lecture03_bacterial_response.csv"
data = pd.read_csv(filename)

data.head()

`pd.read_csv(filename)` reads the comma-separated file and returns a DataFrame. The assignment stores it as `data`.

`.head()` displays the first five rows. It is an inspection method; it does not remove the other rows.

In [ ]:
print("Shape:", data.shape)
print("\nColumns:")
print(data.columns.tolist())
print("\nData types:")
print(data.dtypes)
print("\nMissing values:")
print(data.isna().sum())

The audit confirms the dataset dimensions, column names, pandas data types and missing-value counts.

`shape` is an attribute containing `(rows, columns)`. `.isna()` returns Boolean values, and `.sum()` counts `True` values column by column.

This dataset is synthetic. Each row represents one bacterial sample. The proposed task is to classify the sample's `response` as `sensitive` or `resistant`.

## Data dictionary and possible roles

| Column | Meaning | Initial role |
|---|---|---|
| `sample_id` | unique sample label | identifier; not a model feature |
| `species` | bacterial species | categorical feature |
| `treatment` | treatment group | categorical feature requiring scientific consideration |
| `batch` | laboratory batch | technical metadata and possible feature/confounder |
| `time_h` | collection time | numerical feature |
| `gene_a`, `gene_b`, `gene_c` | normalized expression measures | numerical features |
| `growth_rate` | measured growth rate | numerical feature |
| `response` | sensitive or resistant | classification target |
| `response_code` | numerical copy of response | direct leakage; must be excluded |

A column's role comes from the prediction question and how the information was generated. Data type alone does not determine whether a column is a valid feature.

# Part 3 — Define the prediction question precisely

A modelling question should identify:

- **observation:** one bacterial sample;
- **target:** the outcome to predict;
- **features:** information available when prediction would be made;
- **prediction time:** the point at which the prediction is intended;
- **purpose:** why the prediction is useful.

For this practical:

> Using recorded species, experimental metadata, gene-expression measurements and growth rate, classify each sample as sensitive or resistant.

This sentence is still simplified. In a real project, the timing of every measurement must be considered. A feature measured only after the outcome is known would not support an earlier prediction.

# Part 4 — Features and target

Machine-learning examples commonly use:

- `X` for the feature table;
- `y` for the target labels.

These names come from mathematical conventions. They are variable names, not special Python keywords.

In [ ]:
target_column = "response"

feature_columns = [
    "species",
    "treatment",
    "batch",
    "time_h",
    "gene_a",
    "gene_b",
    "gene_c",
    "growth_rate"
]

X = data[feature_columns].copy()
y = data[target_column].copy()

### Reading the selection code

`target_column` stores one column name as a string.

`feature_columns` is a list containing the approved feature names. Explicitly listing features is safer than dropping only one known target because it prevents forgotten leakage columns from entering automatically.

```python
X = data[feature_columns].copy()
```

Using a list inside the brackets selects several columns and returns a DataFrame. `.copy()` creates a separate object for the workflow.

```python
y = data[target_column].copy()
```

Using one string selects one Series. `y` is therefore one-dimensional.

In [ ]:
print("X shape:", X.shape)
print("y shape:", y.shape)
print("\nFeature columns:")
print(X.columns.tolist())
print("\nTarget counts:")
print(y.value_counts())

If `X` has shape `(120, 8)`, it contains 120 observations and 8 selected features. If `y` has shape `(120,)`, it contains one label for each observation.

The row indexes of `X` and `y` remain aligned. Row index 0 in `X` corresponds to row index 0 in `y`.

# Part 5 — Identifiers are usually not predictive features

`sample_id` is needed for traceability. It allows a result to be connected back to the correct sample. However, an arbitrary label such as `S037` is not a biological measurement.

Including unique identifiers can encourage memorization, particularly with flexible algorithms. A model may learn patterns that do not generalize to new identifiers.

The identifier should therefore be retained separately:

In [ ]:
sample_ids = data["sample_id"].copy()

print(sample_ids.head())

Keeping identifiers outside `X` preserves traceability without offering them to the model.

Not every ID-like column is automatically irrelevant. For example, a site code may carry geographical information. That does not mean it should automatically be included: it requires scientific reasoning about the future population, possible confounding and whether the intended prediction setting will contain known or new sites.

# Part 6 — Data leakage

**Data leakage** occurs when model training uses information that would not legitimately be available for the intended prediction.

The dataset includes `response_code`:

```text
sensitive → 0
resistant → 1
```

This column is simply the target written numerically. If included as a feature, it reveals the answer directly.

In [ ]:
print(pd.crosstab(data["response"], data["response_code"]))

The cross-tabulation shows that `response_code` determines `response`. A model using it would not discover a useful biological pattern; it would read an encoded answer.

Examples of possible leakage include:

- a numerical copy of the target;
- a column created after the outcome occurred;
- a treatment decision made after clinicians knew the diagnosis;
- a filename containing the class label;
- imputation or scaling learned from the complete dataset before splitting;
- samples from the same individual appearing in both training and test sets;
- a test-set label used while choosing model settings.

<div class="decision-box"><b>Leakage is defined by information flow and prediction timing, not only by column names.</b></div>

## A deliberately incorrect feature table

The following code demonstrates what **not** to use:

```python
wrong_X = data.drop(columns=["response"])
```

It removes `response` but silently retains both `sample_id` and `response_code`. “Everything except the target” is not a safe feature-selection rule.

# Part 7 — Why training and test data must be separate

The model learns from the **training set**. The **test set** is kept unseen until final evaluation.

An analogy is studying from one set of questions and then being assessed with different questions. Reporting performance on the study questions does not reveal how well knowledge transfers.

If the same rows are used to fit and evaluate a model, the score can reward memorization.

The test set represents new observations only approximately. Its usefulness depends on whether the split matches the real prediction setting.

# Part 8 — Split the data

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

### Understanding every argument

`train_test_split(...)` receives the aligned feature table `X` and target `y`.

It returns four objects:

- `X_train` — training features;
- `X_test` — test features;
- `y_train` — training labels;
- `y_test` — test labels.

The assignment uses four variable names because the function returns four values in that order.

`test_size=0.25` requests 25% of rows for testing and approximately 75% for training.

`random_state=42` fixes the pseudorandom split. Running the code again with the same data and value produces the same row assignment. The number 42 has no special statistical meaning; it is simply a reproducibility choice.

`stratify=y` asks the split to preserve approximately the same target-class proportions in training and test data.

In [ ]:
print("Training feature shape:", X_train.shape)
print("Test feature shape:", X_test.shape)
print("Training target shape:", y_train.shape)
print("Test target shape:", y_test.shape)

The row counts in `X_train` and `y_train` must match. The row counts in `X_test` and `y_test` must also match.

The indexes are usually no longer consecutive because the original row labels are retained. This is not an error. pandas uses the indexes to preserve row identity.

## Check stratification

In [ ]:
print("Full-data proportions:")
print(y.value_counts(normalize=True).round(3))

print("\nTraining proportions:")
print(y_train.value_counts(normalize=True).round(3))

print("\nTest proportions:")
print(y_test.value_counts(normalize=True).round(3))

`value_counts(normalize=True)` returns proportions instead of counts. `.round(3)` rounds displayed values to three decimal places.

Stratification is helpful for class balance, but it does not solve every splitting problem. Related samples, repeated measurements, subjects, sites, dates or batches may require group-based or time-based splitting. A random row split can be misleading when rows are not independent.

## Verify that row indexes do not overlap

In [ ]:
overlapping_indexes = set(X_train.index) & set(X_test.index)

print("Overlapping row indexes:", overlapping_indexes)

The ampersand `&` calculates the intersection of two sets. An empty set means no row index appears in both partitions.

This check only detects identical row indexes. It does not detect biological relatives, repeated subjects or duplicated measurements represented by different rows.

# Part 9 — Establish a baseline

A model should be compared with a simple reference. For classification, one basic baseline always predicts the most frequent training class.

Without a baseline, an accuracy value can sound impressive even when it adds no value. If 90% of samples belong to one class, always predicting that class gives 90% accuracy.

In [ ]:
baseline = DummyClassifier(strategy="most_frequent")

baseline.fit(X_train, y_train)
baseline_predictions = baseline.predict(X_test)

baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print("Baseline accuracy:", round(baseline_accuracy, 3))

### Reading the baseline code

```python
baseline = DummyClassifier(strategy="most_frequent")
```

creates a classifier object. The strategy tells it to use the most common label observed during fitting.

```python
baseline.fit(X_train, y_train)
```

`.fit()` learns which training label is most common. This dummy model ignores the feature values.

```python
baseline.predict(X_test)
```

creates one prediction for every test row.

`accuracy_score(y_test, baseline_predictions)` compares known labels with predictions and calculates the fraction that match.

# Part 10 — Identify numerical and categorical features

Different data types require different preprocessing.

In [ ]:
numeric_features = [
    "time_h",
    "gene_a",
    "gene_b",
    "gene_c",
    "growth_rate"
]

categorical_features = [
    "species",
    "treatment",
    "batch"
]

The two lists are explicit. Their combined contents should match the columns in `X`.

In [ ]:
listed_features = numeric_features + categorical_features

print("Features in X:", sorted(X.columns.tolist()))
print("Features in lists:", sorted(listed_features))
print("Lists cover X exactly:", set(X.columns) == set(listed_features))

The `+` operator concatenates the two lists. `set(...)` removes ordering and compares membership. `==` asks whether the two sets contain exactly the same names.

This check detects forgotten or extra columns, although it would not detect a duplicated name inside one list. Explicit auditing remains important.

# Part 11 — Numerical preprocessing

The numerical workflow has two steps:

1. replace missing values using the training median;
2. standardize the resulting numerical values.

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]
)

### Understanding `Pipeline`

A pipeline applies steps in order. The `steps` argument receives a list of tuples.

```python
("imputer", SimpleImputer(strategy="median"))
```

is one tuple containing:

- a short step name, `"imputer"`;
- the transformer object that performs the work.

`SimpleImputer(strategy="median")` learns one median per numerical feature during `.fit()`. It later uses those stored medians during `.transform()`.

`StandardScaler()` learns each feature's training mean and standard deviation, then transforms values to standardized units.

The imputer comes first because the scaler cannot calculate meaningful results from missing cells.

# Part 12 — Categorical preprocessing

Categorical labels require numerical representation. The workflow will:

1. replace missing categories with the most frequent training category;
2. create one-hot indicator columns.

In [ ]:
categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ]
)

`strategy="most_frequent"` replaces a missing category with the category that occurs most often in the training column.

`OneHotEncoder` creates one indicator column per learned category.

`handle_unknown="ignore"` is important for future or test data. If an unseen category appears, the encoder does not crash. Its indicator columns for that feature become zero for that row.

Ignoring an unseen category prevents a software error, but it does not guarantee that the prediction is scientifically reliable. A model trained without that category has no direct evidence about it.

# Part 13 — Apply different preprocessing to different columns

In [ ]:
preprocessor = ColumnTransformer(
    transformers=[
        ("numeric", numeric_transformer, numeric_features),
        ("categorical", categorical_transformer, categorical_features)
    ]
)

`ColumnTransformer` routes selected columns to selected transformers.

Each tuple inside `transformers` contains three items:

1. a name;
2. a transformer;
3. a list of columns.

The numerical columns go through median imputation and scaling. The categorical columns go through categorical imputation and one-hot encoding.

The separate transformed outputs are combined into one numerical feature matrix for the classifier.

# Part 14 — Connect preprocessing to the classifier

In [ ]:
model = LogisticRegression(max_iter=1000)

workflow = Pipeline(
    steps=[
        ("preprocessing", preprocessor),
        ("classifier", model)
    ]
)

`LogisticRegression` is a classification algorithm despite the word “regression” in its name. It estimates class probabilities and converts them into class predictions.

`max_iter=1000` allows the optimization procedure up to 1000 iterations. This is a numerical convergence setting, not the number of biological samples.

The complete `workflow` has two major stages:

```text
raw selected columns
        ↓
column-specific preprocessing
        ↓
numerical feature matrix
        ↓
logistic-regression classifier
```

Combining the stages reduces the risk of accidentally preprocessing training and test data inconsistently.

# Part 15 — Fit the workflow on training data only

In [ ]:
workflow.fit(X_train, y_train)

One `.fit()` call performs several learned operations using training data:

1. calculate numerical medians;
2. calculate numerical means and standard deviations;
3. identify categorical levels;
4. identify most frequent categorical values;
5. transform the training features;
6. estimate model parameters.

The test labels and test feature distribution are not used during fitting.

<div class="warning-box"><b>Do not fit the imputer, encoder or scaler on the complete dataset before splitting.</b></div>

## What happens to test data?

When `.predict(X_test)` is called, the workflow uses preprocessing rules already learned from training data:

- training medians fill test missing values;
- training means and standard deviations scale test values;
- training categories define encoded columns;
- trained model parameters produce predictions.

The test set is transformed but not used to relearn those rules.

# Part 16 — Generate predictions

In [ ]:
test_predictions = workflow.predict(X_test)

print("Number of predictions:", len(test_predictions))
print("First five predictions:", test_predictions[:5])

`.predict()` returns one predicted label per test row. `test_predictions[:5]` is a slice containing the first five predictions.

The order matches `X_test`. The predictions can therefore be placed beside test identifiers and known labels.

In [ ]:
prediction_table = pd.DataFrame({
    "sample_id": sample_ids.loc[X_test.index],
    "known_response": y_test,
    "predicted_response": test_predictions
})

prediction_table.head(10)

The dictionary passed to `pd.DataFrame` supplies three output columns.

`sample_ids.loc[X_test.index]` selects identifiers using the exact row indexes assigned to the test set. This restores traceability without using identifiers as model features.

pandas aligns Series by index. The NumPy prediction array is placed in row order. Checking the resulting table is therefore an important verification step.

# Part 17 — Accuracy

In [ ]:
test_accuracy = accuracy_score(y_test, test_predictions)

print("Baseline accuracy:", round(baseline_accuracy, 3))
print("Model test accuracy:", round(test_accuracy, 3))

Accuracy is:

$$
\text{Accuracy} = \frac{\text{number of correct predictions}}{\text{total predictions}}
$$

The model should be compared with the baseline rather than interpreted in isolation.

Accuracy can be misleading when classes are strongly imbalanced or when different errors have different consequences. It also does not measure biological usefulness, transportability or causality.

# Part 18 — Confusion matrix

A confusion matrix counts prediction outcomes by known class and predicted class.

In [ ]:
class_order = ["sensitive", "resistant"]

matrix = confusion_matrix(
    y_test,
    test_predictions,
    labels=class_order
)

matrix_table = pd.DataFrame(
    matrix,
    index=["known_sensitive", "known_resistant"],
    columns=["predicted_sensitive", "predicted_resistant"]
)

matrix_table

The `labels=class_order` argument fixes row and column order. Without an explicit order, misreading the matrix is easy.

Rows represent known labels and columns represent predicted labels in this scikit-learn output.

| Cell | Meaning |
|---|---|
| known sensitive, predicted sensitive | correctly classified sensitive sample |
| known sensitive, predicted resistant | sensitive sample incorrectly called resistant |
| known resistant, predicted sensitive | resistant sample incorrectly called sensitive |
| known resistant, predicted resistant | correctly classified resistant sample |

Whether one error is more serious depends on the intended application.

In [ ]:
plt.figure(figsize=(6, 4))
sns.heatmap(
    matrix_table,
    annot=True,
    fmt="d",
    cmap="Greens",
    cbar=False
)
plt.title("Test-set confusion matrix")
plt.ylabel("Known response")
plt.xlabel("Predicted response")
plt.show()

`annot=True` writes counts inside cells. `fmt="d"` formats them as integers. The heatmap is a display of the same counts, not a new analysis.

# Part 19 — Precision, recall and F1-score

In [ ]:
print(
    classification_report(
        y_test,
        test_predictions,
        labels=class_order,
        zero_division=0
    )
)

For one chosen class:

- **precision** asks: among samples predicted as this class, what proportion were correct?
- **recall** asks: among samples truly belonging to this class, what proportion were found?
- **F1-score** combines precision and recall using their harmonic mean;
- **support** is the number of known test observations in the class.

`zero_division=0` tells scikit-learn to report zero rather than issue an undefined-division warning if no prediction makes a metric calculable.

Metrics must be connected to the question. If missing a resistant sample is especially costly, resistant-class recall may matter more than overall accuracy.

# Part 20 — Training performance and introductory overfitting awareness

Overfitting occurs when a model learns training-specific detail that does not generalize well to unseen observations.

Compare training and test accuracy as an initial diagnostic.

In [ ]:
train_predictions = workflow.predict(X_train)
train_accuracy = accuracy_score(y_train, train_predictions)

print("Training accuracy:", round(train_accuracy, 3))
print("Test accuracy:", round(test_accuracy, 3))
print("Difference:", round(train_accuracy - test_accuracy, 3))

A much higher training score than test score can indicate overfitting, but this comparison is not a complete diagnosis.

- A small difference does not prove generalization.
- A large difference can vary with a small or unlucky test set.
- Similar poor scores can indicate underfitting.
- Leakage can make both scores misleadingly high.

The test set should not become a repeated tuning tool. Repeatedly changing the workflow to improve test performance indirectly trains decisions on the test set.

# Part 21 — Inspect learned transformed feature names

In [ ]:
fitted_preprocessor = workflow.named_steps["preprocessing"]
transformed_feature_names = fitted_preprocessor.get_feature_names_out()

print("Number of transformed features:", len(transformed_feature_names))
print(transformed_feature_names)

`workflow.named_steps` is dictionary-like. The key `"preprocessing"` retrieves the fitted `ColumnTransformer`.

`.get_feature_names_out()` reports names after transformation. They include prefixes such as `numeric__` and `categorical__`, showing which transformer produced each column.

The number of transformed features is larger than the original count because one categorical feature can become several one-hot columns.

# Part 22 — Inspect logistic-regression coefficients cautiously

In [ ]:
fitted_classifier = workflow.named_steps["classifier"]

coefficient_table = pd.DataFrame({
    "feature": transformed_feature_names,
    "coefficient": fitted_classifier.coef_[0]
})

coefficient_table["absolute_coefficient"] = coefficient_table["coefficient"].abs()
coefficient_table = coefficient_table.sort_values(
    "absolute_coefficient",
    ascending=False
)

coefficient_table.head(10)

For a binary logistic-regression model, `coef_[0]` contains one coefficient per transformed feature.

`.abs()` calculates magnitude without sign. Sorting by magnitude shows the strongest fitted associations in this particular model.

Interpretation requires caution:

- coefficients describe associations after preprocessing;
- signs depend on the class order used internally;
- correlated features can divide or destabilize effects;
- one-hot columns are interpreted relative to other category indicators;
- large coefficients do not prove causal importance;
- this synthetic dataset is too small for strong biological conclusions.

Model explanation is not the same as discovering a biological mechanism.

# Part 23 — An incorrect preprocessing workflow and why it leaks

This sequence is incorrect for evaluation:

```python
imputer = SimpleImputer(strategy="median")
all_numbers = imputer.fit_transform(data[numeric_features])

X_train, X_test, y_train, y_test = train_test_split(...)
```

The imputer learns medians from all rows, including future test rows, before splitting. The same problem occurs if scaling, feature selection or category learning is fitted on the complete dataset.

The correct principle is:

```text
split first
fit all learned preprocessing on training data
apply stored preprocessing to test data
```

The pipeline enforces this order when fitted with `X_train` and `y_train`.

# Part 24 — Biological splitting concerns

A random stratified split assumes rows can be assigned independently. Biological datasets often violate this assumption.

Examples:

- several samples from the same patient;
- multiple time points from one culture;
- technical replicates of the same specimen;
- samples from the same field plot;
- batches processed on different days;
- sequences from closely related strains;
- images from the same individual.

If related rows appear in training and test data, the model can exploit shared characteristics and produce an optimistic score.

Possible alternatives include:

- group-aware splitting by patient, organism, culture or plot;
- holding out an entire batch or site;
- time-based splitting when predicting the future;
- external validation using an independent dataset.

The split strategy should imitate the intended use case.

# Part 25 — Save predictions and workflow information

In [ ]:
prediction_table["correct"] = (
    prediction_table["known_response"] ==
    prediction_table["predicted_response"]
)

prediction_table.to_csv(
    "lecture03_test_predictions.csv",
    index=False
)

print("Saved test predictions.")

The comparison creates a Boolean column showing whether each prediction matches the known label.

`.to_csv(..., index=False)` saves the report without adding the pandas index as another column.

Saving predictions allows errors to be traced back to samples and investigated. A summary score alone hides which observations failed.

In [ ]:
reloaded_predictions = pd.read_csv("lecture03_test_predictions.csv")

print("Saved rows:", len(reloaded_predictions))
print("Expected test rows:", len(X_test))
print("Columns:", reloaded_predictions.columns.tolist())

Reopening verifies that the file can be read and has the expected row count and fields.

# Part 26 — Reproducibility checklist

A modelling report should record:

1. prediction question and target definition;
2. observation unit;
3. included and excluded columns;
4. leakage assessment;
5. split method and random seed;
6. training and test sizes;
7. class distributions;
8. preprocessing steps and where they were fitted;
9. baseline definition;
10. model type and settings;
11. evaluation metrics;
12. confusion matrix;
13. error analysis;
14. software versions;
15. limitations and intended population.

Reproducibility is not achieved by writing `random_state=42` alone. The dataset, definitions, code, environment and decisions also matter.

# Part 27 — Guided check 1: identify leakage

Consider these possible features for predicting antibiotic resistance:

1. sample ID;
2. species recorded before testing;
3. gene-expression measurement available at prediction time;
4. final resistance label encoded as 0/1;
5. filename containing `resistant`;
6. laboratory batch;
7. treatment administered after the response result was known.

Classify each as:

- plausible feature;
- exclude;
- requires scientific investigation.

## Suggested discussion

- sample ID: normally exclude from model features but retain for traceability;
- species: potentially legitimate if available and relevant to the intended setting;
- gene expression: potentially legitimate if measured at prediction time;
- encoded final label: direct leakage, exclude;
- label-containing filename: direct leakage, exclude;
- batch: requires investigation because it may represent technical structure or confounding;
- later treatment decision: unavailable at prediction time and therefore leakage for the stated task.

The correct answer depends on when and how information is generated. The reasoning is more important than memorizing column names.

# Part 28 — Guided check 2: change the test proportion

Repeat the split using `test_size=0.30` and `random_state=17`. Keep stratification.

Report:

- training rows;
- test rows;
- training class proportions;
- test class proportions.

Do not overwrite the original split variables; use names ending in `_2`.

In [ ]:
# Write the guided-check code here.

## Guided check 2 — one possible solution

In [ ]:
X_train_2, X_test_2, y_train_2, y_test_2 = train_test_split(
    X,
    y,
    test_size=0.30,
    random_state=17,
    stratify=y
)

print("Training rows:", len(X_train_2))
print("Test rows:", len(X_test_2))
print("\nTraining proportions:")
print(y_train_2.value_counts(normalize=True).round(3))
print("\nTest proportions:")
print(y_test_2.value_counts(normalize=True).round(3))

Changing the seed or test proportion changes which observations appear in each partition. Performance from one split therefore contains sampling variability.

# Part 29 — Guided check 3: manual accuracy

Calculate test accuracy without `accuracy_score`.

### Pseudocode

```text
compare each known label with its prediction
count how many comparisons are True
divide by the total number of predictions
```

In [ ]:
# Write the manual accuracy calculation here.

## Guided check 3 — one possible solution

In [ ]:
correct_predictions = y_test.to_numpy() == test_predictions
number_correct = correct_predictions.sum()
manual_accuracy = number_correct / len(test_predictions)

print("Correct predictions:", number_correct)
print("Total predictions:", len(test_predictions))
print("Manual accuracy:", round(manual_accuracy, 3))

`.to_numpy()` converts the target Series to an array in its current order. The comparison produces Boolean values. Summing Booleans counts `True` values.

The result should match `accuracy_score` apart from displayed rounding.

# Part 30 — Independent practical

Build a fresh classification workflow from the supplied CSV.

## Required tasks

1. State the observation unit and target.
2. Create a table assigning every column one role.
3. Identify and exclude direct leakage.
4. Preserve sample identifiers separately.
5. Create `X` and `y` using an explicit feature list.
6. Make a stratified 80:20 split with a stated seed.
7. Create a majority-class baseline.
8. Build numerical and categorical preprocessing pipelines.
9. Connect them with a logistic-regression classifier.
10. Fit only on training data.
11. predict the test data;
12. report baseline accuracy, model accuracy and a labelled confusion matrix;
13. report precision, recall and F1-score;
14. save sample-level predictions;
15. discuss three limitations.

## Required explanations

Explain in your own words:

- why `sample_id` is retained but excluded from `X`;
- why `response_code` is leakage;
- why splitting occurs before learned preprocessing;
- why `handle_unknown="ignore"` is used;
- why a baseline is required;
- why one test score is not a final claim of general performance.

In [ ]:
# Complete the independent practical in additional cells.

# Part 31 — AI-assisted workflow audit

An AI assistant suggests:

```python
df = pd.read_csv("lecture03_bacterial_response.csv")
X = df.drop(columns="response")
y = df["response"]

X = pd.get_dummies(X).fillna(0)
X_train, X_test, y_train, y_test = train_test_split(X, y)

model = LogisticRegression()
model.fit(X_train, y_train)
print(model.score(X_test, y_test))
```

Audit the proposal before running it.

Questions:

1. Which forbidden features remain in `X`?
2. Which preprocessing steps occur before splitting?
3. Why can filling all missing values with zero be inappropriate?
4. Why should the split use a seed and usually stratification here?
5. What baseline is missing?
6. What information is hidden by reporting one score?
7. How would a pipeline improve the workflow?

AI-generated code should be treated as an unverified proposal. Correct syntax does not guarantee valid scientific methodology.

# Part 32 — Review questions

1. What does one row represent in this dataset?
2. What is the target?
3. What is the difference between `X` and `y`?
4. Why is an explicit feature list safer than dropping only the target?
5. Why is `response_code` leakage?
6. Why is `sample_id` not used as a feature?
7. What is the purpose of a test set?
8. What does `test_size=0.25` mean?
9. What does `random_state` control?
10. What does `stratify=y` attempt to preserve?
11. Why is a baseline necessary?
12. What does a median imputer learn?
13. Why does one-hot encoding increase the number of columns?
14. What does `handle_unknown="ignore"` do?
15. Why should preprocessing be fitted only on training data?
16. What does accuracy measure?
17. What information does a confusion matrix add?
18. What is recall?
19. What can a large training–test score difference suggest?
20. Why can a random row split be inappropriate for biological data?

# Part 33 — Review answers

1. One bacterial sample.
2. `response`, with sensitive and resistant classes.
3. `X` contains predictor features; `y` contains target labels.
4. It prevents forgotten identifiers or leakage columns from entering automatically.
5. It is a numerical copy of the answer being predicted.
6. It is an arbitrary label used for traceability, not a measured predictive property.
7. To evaluate the fitted workflow on observations not used for learning.
8. Twenty-five percent of rows are assigned to the test set.
9. The reproducible pseudorandom row assignment.
10. Approximate target-class proportions.
11. It shows whether the model improves on a trivial prediction rule.
12. One median per fitted numerical feature from training data.
13. Each learned category becomes an indicator feature.
14. It prevents an unseen category from causing a transformation error.
15. Learning preprocessing from test data leaks information into model development.
16. The fraction of predictions equal to the known labels.
17. It separates correct and incorrect predictions by known and predicted class.
18. The proportion of actual members of a class that were identified.
19. Possible overfitting, although more investigation is required.
20. Related or repeated samples may cross partitions and make evaluation optimistic.

# Key terms

| Term | Meaning |
|---|---|
| Observation | One unit represented by a dataset row |
| Feature | Input variable offered to a model |
| Target | Outcome the model is asked to predict |
| Identifier | Label used to trace an observation |
| Training set | Data used to learn preprocessing and model parameters |
| Test set | Held-out data used for evaluation |
| Stratification | Preserving approximate class proportions during splitting |
| Baseline | Simple reference method for comparison |
| Leakage | Illegitimate information entering model development |
| Imputation | Replacement of missing entries using a defined rule |
| One-hot encoding | Conversion of categories into indicator columns |
| Scaling | Transformation of numerical feature magnitudes |
| Pipeline | Ordered object connecting transformations and an estimator |
| Fit | Learn rules or parameters from data |
| Predict | Apply a fitted workflow to produce outputs |
| Accuracy | Fraction of correct predictions |
| Precision | Correct fraction among predictions of a class |
| Recall | Detected fraction among actual members of a class |
| Overfitting | Learning training-specific detail that generalizes poorly |

# Lecture summary

This lecture constructed a complete introductory classification workflow:

- defined the observation and prediction target;
- selected features explicitly;
- excluded identifiers and direct target leakage;
- split data before learned preprocessing;
- preserved class proportions with stratification;
- established a majority-class baseline;
- built separate numerical and categorical preprocessing pipelines;
- combined preprocessing and logistic regression;
- fitted the workflow using training data only;
- generated sample-level test predictions;
- evaluated accuracy, confusion matrix, precision, recall and F1-score;
- compared training and test performance cautiously;
- restored sample identifiers for error analysis;
- saved and verified predictions; and
- considered biological dependence and leakage.

The resulting score is a teaching demonstration, not evidence that the model is ready for biological or clinical use. Later lectures can compare algorithms, strengthen evaluation and examine cross-validation and tuning.